In [ ]:
import numpy as np 
import pandas as pd
import torch.nn as nn
import torch
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import StandardScaler
import sys, os

root = os.path.abspath(os.path.join(os.getcwd(), '..',))

sys.path.insert(0, root)

from utils import time_series_split
from FFN import FCN, FCN_Dropout

In [ ]:
df = pd.read_csv('..\data\BTC_USDT_5m.csv', parse_dates=['open_time'], index_col=['open_time'])
print(df.head())
print(df.info())

In [ ]:
df.isna().sum()

## Scaling Data

In [ ]:
scalar_x = StandardScaler()
scalar_y = StandardScaler()

In [ ]:
x = df[['open', 'high', 'low', 'volume']]
y = df.close


In [ ]:
## inputs: converting numpy default float64(double  precision) to torch default(float32)

x = torch.from_numpy(np.array(x).astype(np.float32))
y = torch.from_numpy(np.array(y).astype(np.float32))


X_train, X_test, y_train, y_test = time_series_split(x, y, .80)

xtrain_scaled = scalar_x.fit_transform(X_train)
ytrain_scaled = scalar_y.fit_transform(y_train.reshape(-1, 1)).flatten()



X_train_scaled = torch.from_numpy(xtrain_scaled.astype(np.float32))
y_train_scaled = torch.from_numpy(ytrain_scaled.astype(np.float32))

## Data Splits

print(X_train_scaled.shape)
print(y_train_scaled.shape)


In [ ]:
## Torch dataset
from torch.utils.data import TensorDataset, DataLoader
dataset = TensorDataset(X_train_scaled, y_train_scaled)

batch_size = 100
train_loader = DataLoader(dataset, batch_size, shuffle=False)
train_loader

### Linear Regression

In [ ]:
## Parameters
learning_rate = 0.001
n_iters = 50
input_size = X_train.shape[1]
output_size = 1

print(input_size)

In [ ]:
## model
reg = nn.Linear(input_size, output_size, dtype=torch.float32)

In [ ]:
## Loss
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(reg.parameters(), learning_rate)


In [ ]:
## Training
reg.train()
for iter in range(n_iters+1):
    for batch_X, batch_y in train_loader:
          
        #Forward pass
        pred = reg(batch_X)
        pred.shape
        loss_value = criterion(pred, batch_y)

        #Backward pass & optimize
        optimizer.zero_grad()
        loss_value.backward()
        optimizer.step()


    if iter % 10 == 0:
        print(f'Epoch {iter}, Loss: {loss_value.item():.4f}')

reg.eval()

### FCN Model

#### Without dropout

In [ ]:
## Parameters
learning_rate = 0.01
n_iters = 50
input_size = X_train.shape[1]
output_size = 1

print(input_size)

In [ ]:
fcn = FCN([4, 12, 8, 1], activation='relu')

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(fcn.parameters(), learning_rate)


In [ ]:
import tensorwatch as tw

watcher = tw.watcher()

In [ ]:
#Training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
fcn.train()
for iter in range(n_iters+1):
    for batch_X, batch_y in train_loader:

        #forward pass
        pred = fcn.forward(batch_X, device)
        loss_value = criterion(pred, batch_y)

        #Backward pass & optimize
        optimizer.zero_grad()
        # torch.nn.utils.clip_grad_norm_(fcn.parameters(), max_norm=1.0)
        loss_value.backward()
        # watcher.observe(fcn, loss=loss_value, gradients=True)
        optimizer.step()
    # watcher.show()
    if iter % 5 == 0:
        print(f'Epoch {iter}, Loss: {loss_value.item():.4f}')

fcn.eval()
# watcher.close()

#### With Dropout

In [ ]:
fcn_dropout = FCN_Dropout([4, 12, 8, 1], dropout_rate=0.2, activation='relu')

In [ ]:
## Parameters
learning_rate = 0.01
n_iters = 50

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(fcn_dropout.parameters(), learning_rate)

In [ ]:
# Training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
fcn_dropout.train()
for iter in range(n_iters+1):
    for batch_X, batch_y in train_loader:
        #forward pass
        pred = fcn_dropout.forward(batch_X, device)
        loss_value = criterion(pred, batch_y)

        #Backward pass & optimize
        optimizer.zero_grad()
        loss_value.backward()
        # torch.nn.utils.clip_grad_norm_(fcn_dropout.parameters(), max_norm=1.0)
        optimizer.step()

    if iter % 5 == 0:
        print(f'Epoch {iter}, Loss: {loss_value.item():.4f}')

fcn_dropout.eval()

### Testing


In [ ]:
X_test_scaled = torch.tensor(scalar_x.transform(X_test), dtype=torch.float32)
y_test_scaled = torch.tensor(scalar_y.transform(y_test.reshape(-1,1)).flatten(), dtype=torch.float32)
y_test_scaled

In [ ]:
##Preditions 
fcn_dropout.eval()
with torch.no_grad():
    predictions_scaled = reg(X_test_scaled).numpy()
    
predictions_scaled

# Inverse transform the scaled predictions
reg_predictions = scalar_y.inverse_transform(predictions_scaled.reshape(-1, 1)).flatten()
reg_predictions

In [ ]:
## Metric
from sklearn.metrics import mean_squared_error

MSE = mean_squared_error(y_test, reg_predictions)
print(f'Regression model MSE on test data: {MSE}')
print(f'Regression model RMSE on test data: {np.sqrt(MSE)}')

In [ ]:
# ## Graph

# # fig, axes = plt.subplots(1, 2, figsize=(20, 6))

# plt.plot(y_test[:100], 'g-', label='original', alpha=.7)
# plt.plot(predictions[:100], 'r-', label='predicted', alpha=.7, linewidth=.8)

# plt.legend()
# plt.show()

## Residual Plotting


In [ ]:
pred = reg_predictions.flatten()
# y_test = y_test.numpy()
print(y_test)
print(pred)


In [ ]:
from scipy import stats
fig, axes = plt.subplots(2, 2, figsize=(15,12))

#Residual vs predictions
residuals = y_test - pred
axes[0,0].scatter(reg_predictions, residuals, alpha=.5)
axes[0,0].set_xlabel('predictions')
axes[0,0].set_ylabel('residuals')
axes[0,0].set_title('Residual vs Predictions')
axes[0, 0].grid(True, alpha=.5)


# Residuals vs Target
axes[0,1].scatter(y_test, residuals, alpha=.5)
axes[0,1].set_xlabel('Targets')
axes[0,1].set_ylabel('residuals')
axes[0,1].set_title('residual vs Target')
axes[0,1].grid(True, alpha=.5)

## distribution of residuals
axes[1,0].hist(residuals, bins=50)
axes[1,0].set_xlabel('Residuals')
axes[1,0].set_ylabel('Frequency')
axes[1,0].set_title('Distribution of Residuals')
axes[1,0].grid(True, alpha=.5)

##Q-Q plot
stats.probplot(residuals, dist='norm', plot=axes[1,1])
axes[1,1].set_title('Q-Q plot')
axes[1,1].grid(True)

plt.tight_layout()
plt.show()